# RAG semantic ICD (bge-m3 + reranker) — build index + A/B trên dev
**Settings**: Accelerator = **GPU T4** · Internet = **On**.

Mục tiêu: đo `candidates_score` khi BẬT lớp vét semantic (RAG) so với lexical thuần.
RAG chỉ chạy khi lexical (synonym+fuzzy) trả rỗng -> chỉ THÊM recall CHẨN_ĐOÁN, không regress.
Extractor giữ **rule** (bản tốt nhất). Nếu J_cand tăng mà text/assert không tụt -> bật semantic khi nộp.

In [ ]:
# 1) Lấy code
%cd /kaggle/working
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /kaggle/working/viettel-medreason
!git pull -q

In [ ]:
# 2) Env — KHÔNG động numpy/torch (bản Kaggle). Thêm FlagEmbedding cho bge-m3 + reranker.
!pip install -q -r requirements.txt
!pip install -q FlagEmbedding
import numpy, torch
print('numpy', numpy.__version__, '| torch', torch.__version__, '| GPU',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not numpy.__version__.startswith('2'):
    raise SystemExit('numpy != 2.x -> Restart & Run All')

In [ ]:
# 3) BUILD index embedding trên KB WHO/BYT (icd10_vn, 11360 mã) — ~1-2 phút trên T4.
#    Sinh data/kb/icd_who_emb.npy + icd_who_meta.parquet (config đã trỏ sẵn 2 path này).
!python src/kb/build_icd_index.py \
  --kb data/kb/icd10_vn.parquet \
  --emb data/kb/icd_who_emb.npy \
  --meta data/kb/icd_who_meta.parquet
!ls -la data/kb/icd_who_emb.npy data/kb/icd_who_meta.parquet

In [ ]:
# 4) Chuẩn bị devset input (60 file có gold WHO)
import os, shutil, glob
os.makedirs('devset/input', exist_ok=True)
for g in glob.glob('data/dev/gold/*.json'):
    n = os.path.basename(g)[:-5]
    shutil.copy(f'data/test/input/{n}.txt', f'devset/input/{n}.txt')
print('devset:', len(glob.glob('devset/input/*.txt')), 'file')

In [ ]:
# 5) A) BASELINE lexical — linking.backend=lexical, extractor=rule
import yaml
def set_backend(b):
    c = yaml.safe_load(open('configs/config.yaml', encoding='utf-8'))
    c['linking']['backend'] = b
    yaml.safe_dump(c, open('configs/config.yaml','w',encoding='utf-8'), allow_unicode=True, sort_keys=False)
set_backend('lexical')
!python src/pipeline.py --input devset/input --output dev_lex --backend rule
print('===== A) LEXICAL =====')
!python src/eval/official_scorer.py --pred dev_lex --gold data/dev/gold

In [ ]:
# 6) B) LEXICAL + RAG semantic — linking.backend=semantic (bge tự tải ~3GB lần đầu)
set_backend('semantic')
!python src/pipeline.py --input devset/input --output dev_sem --backend rule
print('===== B) LEXICAL + SEMANTIC (RAG) =====')
!python src/eval/official_scorer.py --pred dev_sem --gold data/dev/gold
set_backend('lexical')   # trả config về mặc định an toàn

In [ ]:
# 7) So sánh trực tiếp: bao nhiêu concept được RAG vét thêm (lexical rỗng -> semantic điền)
import json, glob, os
def load(d): return {os.path.basename(f): json.load(open(f, encoding='utf-8')) for f in glob.glob(d+'/*.json')}
lex, sem = load('dev_lex'), load('dev_sem')
added = 0; changed_files = 0
for fn in lex:
    la = {(c['text'], tuple(c.get('position', []))): c.get('candidates', []) for c in lex[fn] if c['type']=='CHẨN_ĐOÁN'}
    sa = {(c['text'], tuple(c.get('position', []))): c.get('candidates', []) for c in sem.get(fn, []) if c['type']=='CHẨN_ĐOÁN'}
    d = 0
    for k in la:
        if not la[k] and sa.get(k):
            added += 1; d += 1
    if d: changed_files += 1
print(f'RAG vét thêm candidate cho {added} concept CHẨN_ĐOÁN (trên {changed_files} file) mà lexical bỏ rỗng.')
print('=> Nếu J_cand ở cell 6 > cell 5 mà text/assert không đổi: RAG có giá trị, nên bật khi nộp.')